# Lesson 12B: Variational Autoencoders (VAE) - Practical

Learn probabilistic generative models with VAE.

**Variational Autoencoder (VAE)** learns a probabilistic latent space:
- Encoder outputs μ and σ (mean and std of latent distribution)
- Sample z ~ N(μ, σ²)
- Decoder generates from z
- Loss = Reconstruction + KL divergence

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Loaded!")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.datasets import load_digits

class VAE(nn.Module):
    def __init__(self, input_dim=64, latent_dim=8):
        super().__init__()
        # Encoder
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc_mu = nn.Linear(32, latent_dim)
        self.fc_logvar = nn.Linear(32, latent_dim)
        # Decoder
        self.fc2 = nn.Linear(latent_dim, 32)
        self.fc3 = nn.Linear(32, input_dim)
    
    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        h = F.relu(self.fc2(z))
        return torch.sigmoid(self.fc3(h))
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(x_recon, x, mu, logvar):
    recon_loss = F.mse_loss(x_recon, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kld

# Load and train
digits = load_digits()
X = digits.data / 16.0
X_tensor = torch.FloatTensor(X)

vae = VAE(input_dim=64, latent_dim=8)
optimizer = optim.Adam(vae.parameters(), lr=0.001)

for epoch in range(50):
    x_recon, mu, logvar = vae(X_tensor)
    loss = vae_loss(x_recon, X_tensor, mu, logvar)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/50, Loss: {loss.item():.4f}")

# Generate new samples
with torch.no_grad():
    z_sample = torch.randn(5, 8)
    generated = vae.decode(z_sample)

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i in range(5):
    axes[i].imshow(generated[i].numpy().reshape(8, 8), cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(f'Generated {i+1}')
plt.suptitle('VAE Generated Digits', fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ VAE can generate new digit-like images!")